# 01 — Data Quality and Cleaning

This notebook performs the data-curation stage of the COPD exacerbation forecasting pipeline.

The main objectives are to:

- load the aggregated daily dataset;
- restrict the analysis to the study period (2010–2023);
- verify the integrity of the daily time series;
- validate missing values generated by temporal lag boundaries;
- identify physically implausible atmospheric pressure measurements;
- analyse the distribution and temporal structure of missing values;
- explore whether missingness is associated with the response variable;
- remove observations affected by the maximum lag boundary;
- impute remaining missing environmental values using linear interpolation.

The resulting cleaned dataset is used in the subsequent exploratory analysis and machine-learning stages.

In [1]:
import pandas as pd

from copd_forecasting.config import (
    RAW_DATA_FILE,
    PROCESSED_DATA_DIR,
    DATE_COLUMN,
    STUDY_START,
    STUDY_END,
    MAX_LAG,
)

from copd_forecasting.data import (
    load_daily_data,
    filter_study_period,
    validate_daily_timeline,
    validate_lag_boundary_missingness,
    replace_invalid_pressure_values,
    remove_lag_boundary,
    impute_lagged_features,
    check_remaining_missing_values,
)

In [2]:
from copd_forecasting.config import (
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    DATE_COLUMN,
    STUDY_START,
    STUDY_END,
    MAX_LAG,
)

## 1. Data loading

The original dataset contains several worksheets at different levels of aggregation.  
Only the aggregated daily COPD dataset is required for this forecasting pipeline.

Patient-level and individual attendance-level records are therefore not loaded.

In [3]:
df = load_daily_data(RAW_DATA_FILE)

print(f"Raw dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(
    f"Available date range: "
    f"{df[DATE_COLUMN].min().date()} → "
    f"{df[DATE_COLUMN].max().date()}"
)

Raw dataset shape: 5,478 rows × 187 columns
Available date range: 2010-01-01 → 2024-12-30


## 2. Study period

The analysis is restricted to the period defined for the study, from 1 January 2010 to 31 December 2023.

Observations outside this interval are excluded before further data-quality checks.

In [4]:
df = filter_study_period(df)

print(f"Study dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(
    f"Study period: "
    f"{df[DATE_COLUMN].min().date()} → "
    f"{df[DATE_COLUMN].max().date()}"
)

Study dataset shape: 5,113 rows × 187 columns
Study period: 2010-01-01 → 2023-12-31


## 3. Temporal integrity checks

Before analysing missing values, the temporal structure of the dataset is verified.

First, the daily time series is checked for duplicated or missing dates. Second, missing values in lagged environmental variables are evaluated to confirm that the additional missing observations are exclusively caused by the expected temporal boundary effect.

For a variable with lag \(k\), the expected relationship is:

\[
n_{\text{missing}}(LAG_k) =
n_{\text{missing}}(LAG_0) + k
\]

This distinction is important because boundary missingness is a structural consequence of lag construction rather than a true data-quality problem.

In [5]:
validate_daily_timeline(df)

print("✓ Daily timeline is complete: no duplicated or missing dates.")

✓ Daily timeline is complete: no duplicated or missing dates.


In [6]:
lag_validation = validate_lag_boundary_missingness(df)

n_checks = len(lag_validation)
n_failed = (~lag_validation["valid"]).sum()

print(f"Total lag checks: {n_checks}")
print(f"Failed checks: {n_failed}")

if n_failed == 0:
    print(
        "✓ All additional missing values in lagged variables "
        "are consistent with the expected boundary effect."
    )
else:
    display(
        lag_validation.loc[~lag_validation["valid"]]
    )

Total lag checks: 147
Failed checks: 0
✓ All additional missing values in lagged variables are consistent with the expected boundary effect.


## 4. Physically implausible pressure values

Atmospheric pressure measurements were reviewed for physical plausibility.

Values below **800 hPa** were considered incompatible with surface atmospheric pressure measurements in the geographical context of the study. These observations are treated as missing values and subsequently handled together with the remaining missing environmental data.

Only the anomalous pressure measurements are replaced; all other variables recorded on the same day are preserved.

In [7]:
pressure_lag0_cols = [
    col
    for col in [
        "pres_min_LAG0",
        "pres_med_LAG0",
        "pres_max_LAG0",
    ]
    if col in df.columns
]

invalid_pressure_counts = (
    (df[pressure_lag0_cols] < 800)
    .sum()
    .rename("Invalid values")
)

invalid_pressure_counts

pres_min_LAG0    1
pres_med_LAG0    1
pres_max_LAG0    1
Name: Invalid values, dtype: int64

In [8]:
df = replace_invalid_pressure_values(df)

print("✓ Physically implausible pressure values replaced with NaN.")

✓ Physically implausible pressure values replaced with NaN.


## 5. Missing-data overview

Missingness is analysed using the environmental variables corresponding to the day of exposure (`LAG0`).

Only `LAG0` variables are used for this descriptive analysis because the remaining lagged variables contain the same environmental series shifted in time, together with the structural boundary missingness evaluated above.

In [9]:
lag0_cols = [
    col
    for col in df.columns
    if col.endswith("_LAG0")
]

missing_summary = (
    df[lag0_cols]
    .isna()
    .sum()
    .to_frame("N_missing")
)

missing_summary["Percentage"] = (
    missing_summary["N_missing"]
    / len(df)
    * 100
)

missing_summary = (
    missing_summary
    .sort_values(
        "Percentage",
        ascending=False,
    )
)

missing_summary.loc[
    missing_summary["N_missing"] > 0
].round(3)

,N_missing,Percentage
CO_LAG0,136,2.660
PM2p5_LAG0,48,0.939
O3_LAG0,16,0.313
PM10_LAG0,15,0.293
NO_LAG0,9,0.176
NO2_LAG0,9,0.176
hume_max_LAG0,5,0.098
pres_med_LAG0,3,0.059
pres_min_LAG0,3,0.059
pres_max_LAG0,3,0.059
